# **Module1**

In [ ]:
# ================== CÀI ĐẶT THƯ VIỆN CHO COLAB ==================
# Chạy cell này trước khi chạy code chính

# 1. Cài đặt PyTorch (nếu chưa có hoặc cần update)
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

# 2. Ultralytics (YOLOv11)
!pip install ultralytics

# 3. Transformers + Accelerate (cho BLIP2 và Qwen2.5)
!pip install transformers accelerate

# 4. Pillow (xử lý ảnh)
!pip install Pillow

# 5. OpenCV (đọc video + xử lý frame)
!pip install opencv-python opencv-python-headless

# 6. NumPy (nếu chưa có)
!pip install numpy

# 7. Sentencepiece + Protobuf (cho tokenizer của Qwen)
!pip install sentencepiece protobuf

# 8. BitsAndBytes (để load model 4-bit quantization - tiết kiệm RAM)
!pip install bitsandbytes

# 9. SciPy (dependency cho một số model)
!pip install scipy

# 10. Einops (cho Vision Transformer)
!pip install einops

# 11. xFormers (tăng tốc inference - optional nhưng recommended)
!pip install xformers

# 12. SafeTensors (load model nhanh hơn)
!pip install safetensors

# 13. Hugging Face Hub (download model)
!pip install huggingface-hub

# 14. Flash Attention 2 (optional - tăng tốc attention mechanism)
# Chỉ cài nếu GPU hỗ trợ (A100, H100, RTX 30xx/40xx)
!pip install flash-attn --no-build-isolation
# ======================= M2.2 — CÀI THƯ VIỆN PDF + RAG =======================
!pip -q install pypdf sentence-transformers faiss-cpu



In [ ]:
# ================== CELL 1: IMPORTS & SETUP ==================
"""
Module 1 - Chuẩn bị dữ liệu video và nhận diện đối tượng
Mục đích: Trích keyframe từ video -> Detect objects (YOLO) -> Lưu metadata JSON/CSV
"""

import os
import re
import json
import csv
import gc
import time
from pathlib import Path
from typing import List, Dict, Any, Tuple

import numpy as np
import cv2
from ultralytics import YOLO

# Check GPU
try:
    import torch
    if torch.cuda.is_available():
        print(f"✓ GPU available: {torch.cuda.get_device_name(0)}")
    else:
        print("⚠ GPU not available, using CPU")
except Exception:
    torch = None
    print("⚠ PyTorch not found, using CPU")

print("✓ Imports completed")

In [ ]:
# ================== CELL 2 — CONFIG MODULE 1 ==================

# Thư mục chứa các file .mp4 (bạn sửa lại cho đúng cấu trúc của bạn)
VIDEO_ROOT = Path("/content/drive/MyDrive/videos")   # VD: nơi chứa public_test/videos

# Đường dẫn tới file weights YOLO đã train
WEIGHTS = Path("/content/runs/detect/train/weights/best.pt")

# Thư mục lưu output của Module 1
RUNTIME_DIR   = Path("roadbuddy/runtime")
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)

META_JSON     = RUNTIME_DIR / "metadata.json"       # thông tin từng video
DET_CSV       = RUNTIME_DIR / "detections.csv"      # toàn bộ bbox
SCENE_CACHE   = RUNTIME_DIR / "scene_cache.jsonl"   # để Module 2 gom thành scene

# Tham số YOLO & sampling
CONF_TH       = 0.35       # ngưỡng confidence
FRAME_STEP    = 3          # lấy 1 frame mỗi 3 frame (giảm tải)
MAX_VIDEOS    = None       # None = quét hết, số nguyên = giới hạn số video

print("VIDEO_ROOT :", VIDEO_ROOT)
print("WEIGHTS    :", WEIGHTS)
print("RUNTIME_DIR:", RUNTIME_DIR)


In [ ]:
# ================== CELL 3 — DATA CLASS & HÀM TIỆN ÍCH ==================

@dataclass
class Detection:
    video_name: str
    frame_idx: int
    t_sec: float
    cls_name: str
    conf: float
    x1: int
    y1: int
    x2: int
    y2: int

def list_videos(root: Path) -> List[Path]:
    """
    Quét tất cả file .mp4 trong thư mục VIDEO_ROOT
    """
    root = Path(root)
    vids = []
    for ext in ("*.mp4", "*.avi", "*.mov", "*.mkv"):
        vids.extend(root.rglob(ext))
    vids = sorted(vids)
    print(f"🔍 Found {len(vids)} video(s) under {root}")
    return vids


In [ ]:
# ================== CELL 4 — CHẠY YOLO TRÊN 1 VIDEO ==================

def run_yolo_on_video(model: YOLO, video_path: Path,
                      conf_th: float = CONF_TH,
                      frame_step: int = FRAME_STEP) -> tuple[dict, List[Detection]]:
    """
    Chạy YOLO trên 1 video:
      - Trích frame
      - Detect
      - Trả về:
          meta: dict thông tin video
          dets: list[Detection]
    """
    video_path = Path(video_path)
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        print(f"❌ Không mở được video: {video_path}")
        return {}, []

    fps   = cap.get(cv2.CAP_PROP_FPS) or 25.0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    w     = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
    h     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)
    dur   = total / fps if fps > 0 else 0

    meta = {
        "video_name": video_path.name,
        "video_path": str(video_path),
        "fps": fps,
        "frames": total,
        "width": w,
        "height": h,
        "duration": dur,
    }

    dets: List[Detection] = []
    frame_idx = 0
    t0 = time.time()
    print(f"\n▶️ Video: {video_path.name} | {total} frames | {fps:.1f} FPS | {dur:.1f}s")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx % frame_step != 0:
            frame_idx += 1
            continue

        t_sec = frame_idx / fps if fps > 0 else 0.0

        # Chạy YOLO
        try:
            results = model(frame, verbose=False, conf=conf_th)[0]
        except TypeError:
            # Nếu bản ultralytics khác, thử gọi predict
            results = model.predict(frame, verbose=False, conf=conf_th)[0]

        if results.boxes is not None and len(results.boxes) > 0:
            for box in results.boxes:
                cls_id  = int(box.cls.item())
                conf    = float(box.conf.item())
                x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
                cls_name = str(model.names.get(cls_id, cls_id))

                dets.append(Detection(
                    video_name=video_path.name,
                    frame_idx=frame_idx,
                    t_sec=t_sec,
                    cls_name=cls_name,
                    conf=conf,
                    x1=x1, y1=y1, x2=x2, y2=y2
                ))

        frame_idx += 1

        # In progress cho dễ theo dõi
        if frame_idx % (frame_step * 50) == 0:
            elapsed = time.time() - t0
            print(f"  frame {frame_idx}/{total}  ({frame_idx/total*100:.1f}%)  | {len(dets)} dets  | {elapsed:.1f}s", end="\r")

    cap.release()
    print(f"\n✅ Xong video {video_path.name}: {len(dets)} detections")

    return meta, dets


In [ ]:
# ================== CELL 5 — PIPELINE MODULE 1: CHẠY HẾT & LƯU FILE ==================

def pipeline_module1():
    # 1) Load YOLO model
    assert WEIGHTS.exists(), f"❌ Không thấy file weights: {WEIGHTS}"
    print(f"🚀 Loading YOLO model from: {WEIGHTS}")
    model = YOLO(str(WEIGHTS))
    print("✅ YOLO model loaded.")

    # 2) Liệt kê video
    videos = list_videos(VIDEO_ROOT)
    if MAX_VIDEOS is not None:
        videos = videos[:MAX_VIDEOS]
        print(f"⚠️ Chỉ xử lý {len(videos)} video đầu tiên (MAX_VIDEOS={MAX_VIDEOS})")

    # 3) Chuẩn bị ghi file output
    all_meta = {"videos": []}

    # CSV detections
    f_csv  = open(DET_CSV, "w", newline="", encoding="utf-8")
    writer = csv.writer(f_csv)
    writer.writerow(["video","frame","t_sec","cls","conf","x1","y1","x2","y2"])

    # JSONL scene_cache
    f_sc = open(SCENE_CACHE, "w", encoding="utf-8")

    # 4) Loop qua từng video
    total_dets = 0
    t0_global = time.time()

    for i, vid in enumerate(videos, start=1):
        print(f"\n================= [{i}/{len(videos)}] {vid.name} =================")
        meta, dets = run_yolo_on_video(model, vid)
        if not meta:
            continue

        all_meta["videos"].append(meta)

        # ghi detections vào CSV + scene_cache
        for d in dets:
            writer.writerow([
                d.video_name, d.frame_idx, f"{d.t_sec:.3f}",
                d.cls_name, f"{d.conf:.4f}",
                d.x1, d.y1, d.x2, d.y2
            ])

            rec = {
                "video": d.video_name,
                "frame": d.frame_idx,
                "t": d.t_sec,
                "cls": d.cls_name,
                "conf": d.conf,
                "bbox": [d.x1, d.y1, d.x2, d.y2],
            }
            f_sc.write(json.dumps(rec, ensure_ascii=False) + "\n")

        total_dets += len(dets)
        # giải phóng GPU RAM chút
        gc.collect()

    # 5) Đóng file & lưu metadata
    f_csv.close()
    f_sc.close()

    with open(META_JSON, "w", encoding="utf-8") as f:
        json.dump(all_meta, f, ensure_ascii=False, indent=2)

    elapsed = time.time() - t0_global
    print("\n================= MODULE 1 DONE =================")
    print(f"📁 META_JSON : {META_JSON}")
    print(f"📁 DET_CSV   : {DET_CSV}")
    print(f"📁 SCENE_CACHE: {SCENE_CACHE}")
    print(f"📊 Tổng số detections: {total_dets}")
    print(f"⏱ Thời gian: {elapsed:.1f}s")


# GỌI CHẠY MODULE 1:
pipeline_module1()


# **Module2**


In [ ]:
# ======================= MODULE 2 — COMMON SETUP =======================

from pathlib import Path
from collections import defaultdict, Counter
from typing import List, Dict, Any, Iterable
import os, json, math, gc

# Gốc project (giống Module 1)
RB_ROOT     = "roadbuddy"
RUNTIME_DIR = f"{RB_ROOT}/runtime"
KB_DIR      = f"{RB_ROOT}/kb"

# INPUT từ Module 1
SCENE_CACHE_PATH = f"{RUNTIME_DIR}/scene_cache.jsonl"    # detections theo frame
DET_CSV_PATH     = f"{RUNTIME_DIR}/detections.csv"       # nếu cần dùng thêm

# OUTPUT Module 2.1
SCENE_DESC_PATH  = f"{RUNTIME_DIR}/scene_description.jsonl"

# Tham số gom scene
WIN_SEC   = 3     # độ dài 1 "cảnh" = 3 giây
FPS       = 15    # dùng nếu cần chuyển frame_idx -> thời gian

# Danh sách nhãn cần quan tâm (tùy model YOLO của bạn)
SIGN_NAMES  = ["speed_limit","no_left","no_right","stop","priority","give_way","end_speed","other_sign"]
LIGHT_NAMES = ["red","yellow","green","green_left","green_straight","red_yellow","off"]

# Thư mục và file cho KB luật (RAG)
EMB_MODEL       = "sentence-transformers/all-MiniLM-L6-v2"
RAG_OUT         = f"{KB_DIR}/index"
CHUNKS_JSONL    = f"{RAG_OUT}/chunks.jsonl"
FAISS_INDEX     = f"{RAG_OUT}/faiss.index"

print("RB_ROOT         :", RB_ROOT)
print("RUNTIME_DIR     :", RUNTIME_DIR)
print("SCENE_CACHE_PATH:", SCENE_CACHE_PATH)
print("SCENE_DESC_PATH :", SCENE_DESC_PATH)
print("KB_DIR          :", KB_DIR)
print("RAG_OUT         :", RAG_OUT)


In [ ]:
# ======================= M2.1 — BUILD SCENE DESCRIPTION =======================

def _iter_jsonl(path: str) -> Iterable[Dict[str, Any]]:
    """Đọc từng dòng JSON từ file .jsonl."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Không tìm thấy file: {path}")
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                yield json.loads(line)
            except Exception as e:
                print(f"⚠️ Lỗi parse dòng JSONL: {e}")
                continue

def _window_key(t_sec: float, win_sec: int) -> int:
    """Chia trục thời gian thành các cửa sổ dài win_sec giây."""
    return int(t_sec // win_sec)

def _summarize_window(items: List[Dict[str, Any]]) -> Dict[str, Any]:
    """
    Gom nhiều detection trong 1 cửa sổ thành mô tả tổng quát.
    items: list dict {video, frame, t, cls, conf, bbox}
    """
    if not items:
        return {
            "signs": [],
            "lights": [],
            "raw_classes": [],
            "desc": "Không có đối tượng đặc biệt."
        }

    cls_list = [it.get("cls", "") for it in items]
    cls_cnt  = Counter(cls_list)

    signs  = [c for c in cls_cnt.keys() if c in SIGN_NAMES]
    lights = [c for c in cls_cnt.keys() if c in LIGHT_NAMES]

    parts = []
    if signs:
        parts.append("Có các biển báo: " + ", ".join(signs))
    if lights:
        parts.append("Đèn tín hiệu xuất hiện: " + ", ".join(lights))
    if not parts:
        parts.append("Không phát hiện biển báo hoặc đèn tín hiệu rõ ràng.")

    desc = " ".join(parts)

    return {
        "signs": signs,
        "lights": lights,
        "raw_classes": list(cls_cnt.items()),
        "desc": desc
    }

def build_scene_description(
    scene_cache_path: str = SCENE_CACHE_PATH,
    scene_desc_path: str = SCENE_DESC_PATH,
    win_sec: int = WIN_SEC
):
    """
    Đọc scene_cache.jsonl (Module 1) → gom theo cửa sổ WIN_SEC giây
    → ghi scene_description.jsonl cho Module 3 dùng.
    """
    scene_cache_path = Path(scene_cache_path)
    scene_desc_path  = Path(scene_desc_path)
    scene_desc_path.parent.mkdir(parents=True, exist_ok=True)

    # Gom detection theo (video, window_id)
    buckets: Dict[str, Dict[int, List[Dict[str, Any]]]] = defaultdict(lambda: defaultdict(list))

    print(f"🔍 Đang đọc scene_cache từ: {scene_cache_path}")
    for rec in _iter_jsonl(scene_cache_path):
        vid  = rec.get("video")
        t    = float(rec.get("t", 0.0))
        win_id = _window_key(t, win_sec)
        buckets[vid][win_id].append(rec)

    # Ghi file scene_description
    wrote = 0
    with open(scene_desc_path, "w", encoding="utf-8") as f_out:
        for vid, win_map in buckets.items():
            for win_id, items in sorted(win_map.items(), key=lambda x: x[0]):
                if not items:
                    continue
                t0 = min(float(x.get("t", 0.0)) for x in items)
                t1 = max(float(x.get("t", 0.0)) for x in items)

                summary = _summarize_window(items)
                out = {
                    "video": vid,
                    "win_id": win_id,
                    "t0": t0,
                    "t1": t1,
                    "signs": summary["signs"],
                    "lights": summary["lights"],
                    "raw_classes": summary["raw_classes"],
                    "desc": summary["desc"]
                }
                f_out.write(json.dumps(out, ensure_ascii=False) + "\n")
                wrote += 1

    print(f"✅ Đã tạo scene_description: {scene_desc_path} | {wrote} dòng")

# CHẠY M2.1
build_scene_description()


In [ ]:
# ======================= M2.3 — CHUYỂN PDF LUẬT → TXT =======================
from pypdf import PdfReader

def pdf_to_txt(input_pdf: str, output_txt: str = None) -> str:
    """
    Đọc toàn bộ text trong file PDF và ghi ra file .txt (UTF-8).
    input_pdf : đường dẫn file .pdf gốc (luật giao thông, nghị định,…)
    output_txt: đường dẫn file .txt xuất ra. Nếu None -> đổi đuôi .pdf -> .txt
    """
    in_path = Path(input_pdf)
    assert in_path.exists(), f"❌ Không tìm thấy file PDF: {in_path}"

    if output_txt is None:
        output_txt = in_path.with_suffix(".txt")
    out_path = Path(output_txt)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    reader = PdfReader(str(in_path))
    texts = []

    print(f"📖 Đang đọc PDF: {in_path} | {len(reader.pages)} trang")
    for i, page in enumerate(reader.pages):
        try:
            t = page.extract_text() or ""
        except Exception as e:
            print(f"⚠️ Lỗi đọc trang {i}: {e}")
            t = ""
        texts.append(t.strip() + "\n")

    full_text = "\n".join(texts)
    out_path.write_text(full_text, encoding="utf-8")
    print(f"✅ Đã ghi TXT: {out_path} | {len(full_text)} ký tự")

    return str(out_path)

def convert_all_pdfs_in_folder(pdf_dir: str, out_dir: str = None) -> List[str]:
    """
    Quét tất cả file .pdf trong pdf_dir, mỗi file tạo 1 file .txt tương ứng.
    Trả về list đường dẫn .txt.
    """
    pdf_dir = Path(pdf_dir)
    assert pdf_dir.exists(), f"❌ Không tìm thấy thư mục PDF: {pdf_dir}"

    if out_dir is None:
        out_dir = pdf_dir
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    txt_files = []
    for pdf_path in pdf_dir.rglob("*.pdf"):
        rel = pdf_path.relative_to(pdf_dir)
        out_txt = out_dir / rel
        out_txt = out_txt.with_suffix(".txt")
        out_txt.parent.mkdir(parents=True, exist_ok=True)
        txt_path = pdf_to_txt(str(pdf_path), str(out_txt))
        txt_files.append(txt_path)

    print(f"✅ Đã convert {len(txt_files)} file PDF → TXT")
    return txt_files


In [ ]:
# ======================= M2.4 — BUILD & LOAD KB LUẬT (RAG) =======================
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

def build_law_kb_from_text_files(
    files: List[str],
    emb_model_name: str = EMB_MODEL,
    out_dir: str = RAG_OUT,
    chunks_jsonl: str = CHUNKS_JSONL,
    faiss_index_path: str = FAISS_INDEX
):
    """
    Đọc 1 hoặc nhiều file .txt luật giao thông → tách thành các đoạn (chunk)
    → embed → build FAISS index + lưu chunks.jsonl.
    """
    files = [str(f) for f in files]
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    # 1) Đọc & tách chunk
    chunks = []
    for fp in files:
        p = Path(fp)
        if not p.exists():
            print(f"⚠️ Không thấy file luật: {p}")
            continue
        txt = p.read_text(encoding="utf-8", errors="ignore")
        for line in txt.splitlines():
            line = line.strip()
            if not line:
                continue
            chunks.append({
                "source": str(p),
                "text": line
            })

    if not chunks:
        raise RuntimeError("❌ Không thu được chunk nào từ file luật.")

    print(f"📚 Số đoạn luật (chunks): {len(chunks)}")

    # 2) Embed bằng SentenceTransformer
    print(f"🔢 Đang embed bằng model: {emb_model_name}")
    model = SentenceTransformer(emb_model_name)
    texts = [c["text"] for c in chunks]
    embeds = model.encode(texts, show_progress_bar=True, convert_to_numpy=True)
    embeds = embeds.astype("float32")

    # 3) Lưu chunks.jsonl
    with open(chunks_jsonl, "w", encoding="utf-8") as f:
        for c in chunks:
            f.write(json.dumps(c, ensure_ascii=False) + "\n")
    print(f"✅ Đã lưu chunks: {chunks_jsonl}")

    # 4) Build FAISS index
    dim = embeds.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(embeds)
    faiss.write_index(index, str(faiss_index_path))
    print(f"✅ Đã lưu FAISS index: {faiss_index_path}")


def load_law_kb(
    emb_model_name: str = EMB_MODEL,
    chunks_jsonl: str = CHUNKS_JSONL,
    faiss_index_path: str = FAISS_INDEX
):
    """
    Load model embed + chunks + FAISS index → phục vụ search_law_kb ở Module 3.
    """
    model = SentenceTransformer(emb_model_name)

    chunks = []
    with open(chunks_jsonl, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                chunks.append(json.loads(line))

    index = faiss.read_index(str(faiss_index_path))
    print(f"✅ Loaded KB: {len(chunks)} chunks")

    return model, chunks, index


def search_law_kb(query: str, model, chunks, index, top_k: int = 5) -> List[Dict[str, Any]]:
    """
    Tìm top_k đoạn luật gần nhất với câu query.
    """
    q_emb = model.encode([query], convert_to_numpy=True).astype("float32")
    D, I = index.search(q_emb, top_k)
    idxs = I[0].tolist()
    return [chunks[i] for i in idxs]


print("✅ Module 2 (Scene + PDF→TXT + KB Luật) đã sẵn sàng.")


In [ ]:
# ======================= M2.5 — VÍ DỤ: CHUYỂN PDF LUẬT + BUILD KB =======================

# 1) Đường dẫn PDF luật (SỬA LẠI cho đúng file của bạn)
LUAT_PDF = "/content/legal-ai/raw/luatgiaothong.pdf"
LUAT_TXT = "/content/legal-ai/data_clean/luatgiaothong.txt"

# 2) PDF -> TXT
LUAT_TXT = pdf_to_txt(LUAT_PDF, LUAT_TXT)

# 3) Build KB từ TXT
build_law_kb_from_text_files([LUAT_TXT])

# 4) Load & test search
model_law, chunks_law, index_law = load_law_kb()
hits = search_law_kb(
    "quy định khi gặp biển cấm rẽ trái và đèn tín hiệu đỏ",
    model_law, chunks_law, index_law, top_k=3
)
for h in hits:
    print("----")
    print(h["text"])


# **Modulle3**

In [ ]:
# ======================= MODULE 3 — COMMON SETUP =======================

from pathlib import Path
import json, re, time, gc
from typing import List, Dict, Any
import pandas as pd

# Gốc project như các module trước
RB_ROOT     = "roadbuddy"
RUNTIME_DIR = f"{RB_ROOT}/runtime"
KB_DIR      = f"{RB_ROOT}/kb"

# INPUT
PUBLIC_TEST_JSON = "/content/data/public_test.json"     # 👉 SỬA THÀNH FILE public_test.json CỦA BẠN
SCENE_DESC_PATH  = f"{RUNTIME_DIR}/scene_description.jsonl"

# KB luật (Module 2)
RAG_OUT         = f"{KB_DIR}/index"
CHUNKS_JSONL    = f"{RAG_OUT}/chunks.jsonl"
FAISS_INDEX     = f"{RAG_OUT}/faiss.index"

# OUTPUT
OUT_CSV         = f"{RUNTIME_DIR}/answers_public_test.csv"

print("PUBLIC_TEST_JSON:", PUBLIC_TEST_JSON)
print("SCENE_DESC_PATH :", SCENE_DESC_PATH)
print("CHUNKS_JSONL    :", CHUNKS_JSONL)
print("FAISS_INDEX     :", FAISS_INDEX)
print("OUT_CSV         :", OUT_CSV)


In [ ]:
# ======================= M3.1 — LOAD CÂU HỎI & SCENE DESCRIPTION =======================

def load_public_test(path: str) -> List[Dict[str, Any]]:
    """
    Đọc file public_test.json → trả về list dict
    Giả định format: [{ "id": ..., "q": ..., "opts": [...], "video": ...}, ...]
    """
    p = Path(path)
    assert p.exists(), f"❌ Không tìm thấy PUBLIC_TEST_JSON: {p}"
    data = json.loads(p.read_text(encoding="utf-8"))
    print(f"✅ Loaded public_test: {len(data)} câu hỏi")
    return data

def load_scene_description(path: str) -> Dict[str, Dict[str, Any]]:
    """
    Đọc scene_description.jsonl → gom theo video base name.
    Trả về: map[video_name] = {
        "descs": [desc1, desc2, ...],
        "signs": set(...),
        "lights": set(...),
        "windows": [ {video,win_id,t0,t1,desc,...}, ... ]
    }
    """
    p = Path(path)
    assert p.exists(), f"❌ Không tìm thấy SCENE_DESC_PATH: {p}"

    scene_map: Dict[str, Dict[str, Any]] = {}

    with p.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
            except Exception:
                continue

            vid  = rec.get("video")
            base = Path(vid).name
            desc = rec.get("desc", "")
            signs = rec.get("signs", [])
            lights = rec.get("lights", [])

            if base not in scene_map:
                scene_map[base] = {
                    "descs": [],
                    "signs": set(),
                    "lights": set(),
                    "windows": []
                }

            if desc:
                scene_map[base]["descs"].append(desc)
            for s in signs:
                scene_map[base]["signs"].add(s)
            for l in lights:
                scene_map[base]["lights"].add(l)
            scene_map[base]["windows"].append(rec)

    print(f"✅ Loaded scene_description cho {len(scene_map)} video")
    return scene_map

# Test load
public_items = load_public_test(PUBLIC_TEST_JSON)
scene_map    = load_scene_description(SCENE_DESC_PATH)


In [ ]:
# ======================= M3.2 — LOAD KB LUẬT (RAG) =======================

# Dùng lại hàm load_law_kb, search_law_kb đã định nghĩa ở Module 2
# Nếu bạn chạy notebook liên tục thì hàm đó đang còn trong RAM
# Nếu không, import lại hoặc copy đoạn code Module 2.4 vào trước.

model_law, chunks_law, index_law = load_law_kb(
    emb_model_name=EMB_MODEL,
    chunks_jsonl=CHUNKS_JSONL,
    faiss_index_path=FAISS_INDEX
)

def fetch_laws_for_question(q_text: str, scene_text: str, top_k: int = 5) -> List[str]:
    """
    Ghép câu hỏi + mô tả cảnh → query KB → trả về list text luật (top_k).
    """
    query = q_text
    if scene_text:
        query = q_text + " | " + scene_text

    hits = search_law_kb(query, model_law, chunks_law, index_law, top_k=top_k)
    return [h["text"] for h in hits]


In [ ]:
# ======================= M3.3 — BUILD CONTEXT CHO TỪNG CÂU HỎI =======================

def _normalize_video_name(video_field: str) -> str:
    """
    Lấy base name từ trường 'video' trong JSON để map sang scene_map.
    Ví dụ: 'public_test/videos/abc123.mp4' -> 'abc123.mp4'
    """
    return Path(video_field).name

def build_context_for_question(item: Dict[str, Any], scene_map: Dict[str, Dict[str, Any]]) -> Dict[str, Any]:
    """
    Từ 1 câu hỏi và scene_map → tạo context:
      - scene_text: ghép các desc của video
      - labels   : list các nhãn (biển + đèn) phát hiện được
      - ocr      : tạm thời [] (nếu chưa có Module OCR)
      - temp     : tạm thời [] (nếu chưa có logic temporal)
      - laws     : list câu luật text liên quan (RAG)
    """
    qid   = item.get("id")
    qtext = item.get("q", "")
    video = item.get("video", "")
    base  = _normalize_video_name(video)

    scene_entry = scene_map.get(base, None)
    if scene_entry is None:
        scene_text = ""
        labels     = []
    else:
        # ghép tất cả mô tả cảnh thành 1 đoạn text
        descs = scene_entry["descs"]
        scene_text = " ".join(descs)
        # labels: union signs + lights
        labels = list(scene_entry["signs"] | scene_entry["lights"])

    # OCR + temp chưa có → để trống
    ocr  = []
    temp = []

    # Lấy luật liên quan từ KB
    laws = fetch_laws_for_question(qtext, scene_text, top_k=5)

    ctx = {
        "qid": qid,
        "qtext": qtext,
        "video": video,
        "scene_text": scene_text,
        "labels": labels,
        "ocr": ocr,
        "temp": temp,
        "laws": laws,
    }
    return ctx

# Test thử build context 1 câu
ctx0 = build_context_for_question(public_items[0], scene_map)
ctx0


In [ ]:
# ======================= M3.4 — HÀM TRẢ LỜI 1 CÂU HỎI =======================

def answer_one(item: Dict[str, Any], scene_map: Dict[str, Dict[str, Any]]) -> Dict[str, Any]:
    """
    Trả lời 1 câu hỏi:
      - Xây context từ scene + luật
      - Gọi infer_phi(...)
      - Trả về dict: {id, answer, explain}
    """
    qid    = item.get("id")
    qtext  = item.get("q", "")
    options = item.get("opts", [])
    if not options:
        options = ["A", "B", "C", "D"]  # fallback cho đủ

    ctx = build_context_for_question(item, scene_map)

    # infer_phi(q, options, scene, labels, ocr, laws, temp)
    ans_letter, explanation = infer_phi(
        ctx["qtext"],
        options,
        ctx["scene_text"],
        ctx["labels"],
        ctx["ocr"],
        ctx["laws"],
        ctx["temp"]
    )

    return {
        "id": qid,
        "answer": ans_letter,
        "explain": explanation,
        "video": ctx["video"]
    }

# Test 1 câu
test_res = answer_one(public_items[0], scene_map)
test_res


In [ ]:
# ======================= M3.5 — CHẠY TOÀN BỘ & XUẤT CSV =======================

def run_module3(
    public_test_json: str = PUBLIC_TEST_JSON,
    scene_desc_path: str = SCENE_DESC_PATH,
    out_csv: str = OUT_CSV
):
    items = load_public_test(public_test_json)
    scene_map = load_scene_description(scene_desc_path)

    rows = []
    t0 = time.time()

    for i, it in enumerate(items, start=1):
        try:
            res = answer_one(it, scene_map)
            rows.append({"id": res["id"], "answer": res["answer"]})
            # log nhẹ
            if i % 10 == 0 or i == len(items):
                print(f"[{i}/{len(items)}] {res['id']}: {res['answer']}  ({res['video']})")
        except Exception as e:
            print(f"❌ Lỗi ở câu {it.get('id')}: {e}")
            rows.append({"id": it.get("id"), "answer": "A"})  # fallback
            gc.collect()

    df = pd.DataFrame(rows)
    out_path = Path(out_csv)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_path, index=False)

    elapsed = time.time() - t0
    print("==============================================")
    print(f"✅ MODULE 3 DONE - Saved CSV: {out_path}")
    print(f"🔢 Số câu: {len(rows)}")
    print(f"⏱ Thời gian: {elapsed:.1f}s")

# GỌI CHẠY MODULE 3
run_module3()
